# 4 · Evaluate (GPU job) + view results
Scores base vs SFT vs SFT+GRPO on the held-out split with the same verifier used for the reward; uploads metrics + chart to `{results_s3}`, then downloads and shows them.

In [ ]:
import json
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, StoppingCondition

sess = Session()
cfg = json.load(open('toolcall_config.json'))
region = cfg['region']
INSTANCE_TYPE = cfg['gpu_instance']
image_uri = cfg['gpu_image_uri'] or image_uris.retrieve(
    framework='pytorch', region=region, version=cfg['pt_version'],
    instance_type=INSTANCE_TYPE, image_scope='training')
print('Region  :', region)
print('Instance:', INSTANCE_TYPE)
print('Image   :', image_uri)

In [ ]:
environment = {
    'BASE_MODEL': cfg['base_model'],
    'DATA_S3': cfg['data_s3'],
    'SFT_MODEL_S3': cfg['sft_model_s3'],
    'GRPO_MODEL_S3': cfg['grpo_model_s3'],
    'OUTPUT_RESULTS_S3': cfg['results_s3'],
    'SEQ_LEN': str(cfg['seq_len']), 'SEED': str(cfg['seed']),
    'REWARD_WEIGHTS': cfg['reward_weights'], 'EVAL_BASE': str(cfg['eval_base']),
    'EVAL_MAX_NEW_TOKENS': str(cfg['eval_max_new_tokens']), 'EVAL_BATCH_SIZE': str(cfg['eval_batch_size']),
    'HF_HOME': '/opt/ml/hf_home', 'HF_HUB_ENABLE_HF_TRANSFER': '1', 'TOKENIZERS_PARALLELISM': 'false',
}
if cfg.get('hf_token'):
    environment['HF_TOKEN'] = cfg['hf_token']
    environment['HUGGING_FACE_HUB_TOKEN'] = cfg['hf_token']
source_code = SourceCode(source_dir='../src', requirements='requirements.txt', entry_script='evaluate.sh')
trainer = ModelTrainer(sagemaker_session=sess, role=cfg['role'], training_image=image_uri,
    source_code=source_code, compute=Compute(instance_type=INSTANCE_TYPE, instance_count=1),
    environment=environment, base_job_name='toolcall-eval',
    stopping_condition=StoppingCondition(max_runtime_in_seconds=12*3600))
trainer.train(wait=True)
print('results ->', cfg['results_s3'])

### Download + display results

In [ ]:
import boto3, json, os
from urllib.parse import urlparse
u = urlparse(cfg['results_s3']); bucket, prefix = u.netloc, u.path.strip('/')
s3 = boto3.client('s3', region_name=region)
os.makedirs('results', exist_ok=True)
for key in ('metrics.json', 'metrics.csv', 'comparison.png', 'samples.json'):
    try:
        s3.download_file(bucket, f'{prefix}/{key}', f'results/{key}')
    except Exception as e:
        print('skip', key, e)
import pandas as pd
display(pd.DataFrame(json.load(open('results/metrics.json'))).set_index('model').round(4))

In [ ]:
from IPython.display import Image
Image('results/comparison.png')